# NBA Over/Under Model Analysis
**Models:** Random Forest (M1) · Logistic Regression (M2) · KNN K=27 (M3)  
**Season:** 2025-26  
**Target:** OVER/UNDER player's rolling 10-game PTS+REB+AST average

This notebook evaluates all three trained models using:
- ROC curves & AUC scores
- Confusion matrices
- Classification reports (precision, recall, F1)
- Feature importance (RF) & coefficients (LR)
- Probability calibration
- Model comparison summary

In [2]:
import pandas as pd
import numpy as np
import glob
import pickle
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    roc_curve, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
})

print('Libraries loaded.')

Libraries loaded.


## Step 1: Load Data & Models


In [3]:
# ── Load models ───────────────────────────────────────────────────────────
m1 = pickle.load(open('finalized_model_M1.sav', 'rb'))
m2 = pickle.load(open('finalized_model_M2.sav', 'rb'))
m3 = pickle.load(open('finalized_model_M3.sav', 'rb'))

m1_features = list(m1.feature_names_in_)
m2_features = m2['features']
m3_features = m3['features']
best_k      = m3['k']

print('Models loaded:')
print(f'  M1 (Random Forest)       — {len(m1_features)} features')
print(f'  M2 (Logistic Regression) — {len(m2_features)} features: {m2_features}')
print(f'  M3 (KNN, K={best_k})         — {len(m3_features)} features')

FileNotFoundError: [Errno 2] No such file or directory: 'finalized_model_M1.sav'

In [ ]:
# ── Load CSV data ─────────────────────────────────────────────────────────
files = glob.glob('nba_player_gamelogs_2025-26.csv')
if not files:
    files = glob.glob('nba_player_gamelogs_*.csv')
print(f'Files found: {files}')

dfs = [pd.read_csv(f) for f in sorted(files)]
df_raw = pd.concat(dfs, ignore_index=True)
print(f'Raw shape: {df_raw.shape}')

# Numeric coercion
num_cols = ['MIN','FGM','FGA','FG_PCT','FG3M','FG3A','FG3_PCT',
            'FTM','FTA','FT_PCT','OREB','DREB','REB','AST',
            'TOV','STL','BLK','BLKA','PF','PFD','PTS','PLUS_MINUS','NBA_FANTASY_PTS']
for col in num_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

df_raw['WL'] = df_raw['WL'].map({'W': 1, 'L': 0})
df_raw = df_raw.dropna(subset=['PTS','REB','AST','MIN']).reset_index(drop=True)
print(f'Cleaned shape: {df_raw.shape}')

In [ ]:
# ── Recreate target variable ──────────────────────────────────────────────
df_raw['COMBO_SCORE'] = df_raw['PTS'] + df_raw['REB'] + df_raw['AST']

player_col = 'Player_ID' if 'Player_ID' in df_raw.columns else 'PLAYER_ID'
df_raw = df_raw.sort_values([player_col]).reset_index(drop=True)

df_raw['ROLLING_AVG'] = (
    df_raw.groupby(player_col)['COMBO_SCORE']
          .transform(lambda x: x.shift(1).rolling(window=10, min_periods=3).mean())
)

df = df_raw.dropna(subset=['ROLLING_AVG']).reset_index(drop=True)
df['TARGET'] = (df['COMBO_SCORE'] > df['ROLLING_AVG']).astype(int)

print(f'Dataset shape: {df.shape}')
print(f'Class balance: {df["TARGET"].mean():.2%} OVER')

In [ ]:
# ── Build feature matrices for each model ─────────────────────────────────
drop_cols = ['TARGET', 'COMBO_SCORE', 'ROLLING_AVG', 'PTS', 'REB', 'AST']

# Fill all needed columns
for col in m1_features + m2_features:
    if col not in df.columns:
        df[col] = 0

X_m1 = df[m1_features].fillna(0)
X_m2 = df[m2_features].fillna(0)
X_m3 = df[m3_features].fillna(0)
Y    = df['TARGET']

# Use same random state as training
_, X_test_m1, _, Y_test = train_test_split(X_m1, Y, test_size=0.2, random_state=42, stratify=Y)
_, X_test_m2, _, _      = train_test_split(X_m2, Y, test_size=0.2, random_state=42, stratify=Y)
_, X_test_m3, _, _      = train_test_split(X_m3, Y, test_size=0.2, random_state=42, stratify=Y)

# Scale for M2 and M3
X_test_m2_scaled = m2['scaler'].transform(X_test_m2)
X_test_m3_scaled = m3['scaler'].transform(X_test_m3)

print(f'Test set size: {len(Y_test):,} rows')
print(f'Test OVER%: {Y_test.mean():.2%}')

## Step 2: Predictions & Probabilities


In [ ]:
# Generate predictions and probabilities
proba_m1 = m1.predict_proba(X_test_m1)[:, 1]
proba_m2 = m2['model'].predict_proba(X_test_m2_scaled)[:, 1]
proba_m3 = m3['model'].predict_proba(X_test_m3_scaled)[:, 1]
proba_ens = (proba_m1 + proba_m2 + proba_m3) / 3

pred_m1  = m1.predict(X_test_m1)
pred_m2  = m2['model'].predict(X_test_m2_scaled)
pred_m3  = m3['model'].predict(X_test_m3_scaled)
pred_ens = (proba_ens >= 0.5).astype(int)

acc_m1  = (pred_m1  == Y_test).mean()
acc_m2  = (pred_m2  == Y_test).mean()
acc_m3  = (pred_m3  == Y_test).mean()
acc_ens = (pred_ens == Y_test).mean()

auc_m1  = roc_auc_score(Y_test, proba_m1)
auc_m2  = roc_auc_score(Y_test, proba_m2)
auc_m3  = roc_auc_score(Y_test, proba_m3)
auc_ens = roc_auc_score(Y_test, proba_ens)

print(f'{"Model":<30} {"Accuracy":>10} {"AUC":>10}')
print('-' * 52)
print(f'{"Random Forest (M1)":<30} {acc_m1:>10.4f} {auc_m1:>10.4f}')
print(f'{"Logistic Regression (M2)":<30} {acc_m2:>10.4f} {auc_m2:>10.4f}')
print(f'{f"KNN K={best_k} (M3)":<30} {acc_m3:>10.4f} {auc_m3:>10.4f}')
print(f'{"Ensemble":<30} {acc_ens:>10.4f} {auc_ens:>10.4f}')

## Step 3: ROC Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── ROC curves ────────────────────────────────────────────────────────────
ax = axes[0]
for proba, label, color in [
    (proba_m1,  f'Random Forest   (AUC={auc_m1:.3f})',  'steelblue'),
    (proba_m2,  f'Logistic Reg.   (AUC={auc_m2:.3f})',  'tomato'),
    (proba_m3,  f'KNN K={best_k}       (AUC={auc_m3:.3f})',  'seagreen'),
    (proba_ens, f'Ensemble        (AUC={auc_ens:.3f})',  'darkorchid'),
]:
    fpr, tpr, _ = roc_curve(Y_test, proba)
    ax.plot(fpr, tpr, label=label, linewidth=2, color=color)

ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Random baseline')
ax.fill_between([0,1],[0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(fontsize=9)

# ── Accuracy + AUC bar chart ───────────────────────────────────────────────
ax2 = axes[1]
labels  = ['RF (M1)', 'LR (M2)', f'KNN (M3)', 'Ensemble']
accs    = [acc_m1, acc_m2, acc_m3, acc_ens]
aucs    = [auc_m1, auc_m2, auc_m3, auc_ens]
x       = np.arange(len(labels))
width   = 0.35
colors  = ['steelblue', 'tomato', 'seagreen', 'darkorchid']

bars1 = ax2.bar(x - width/2, accs, width, label='Accuracy', alpha=0.85, color=colors, edgecolor='white')
bars2 = ax2.bar(x + width/2, aucs, width, label='AUC',      alpha=0.55, color=colors, edgecolor='white', hatch='//')

for bar, val in zip(bars1, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')
for bar, val in zip(bars2, aucs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.set_ylim(min(accs + aucs) - 0.05, max(accs + aucs) + 0.05)
ax2.set_ylabel('Score')
ax2.set_title('Accuracy vs AUC Comparison', fontweight='bold')
ax2.legend()

plt.suptitle('NBA Over/Under Model Performance — 2025-26 Season', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Step 4: Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, preds, title, color in zip(
    axes,
    [pred_m1, pred_m2, pred_m3, pred_ens],
    ['Random Forest (M1)', 'Logistic Reg (M2)', f'KNN K={best_k} (M3)', 'Ensemble'],
    ['Blues', 'Reds', 'Greens', 'Purples']
):
    cm = confusion_matrix(Y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['UNDER', 'OVER'])
    disp.plot(ax=ax, colorbar=False, cmap=color)
    ax.set_title(title, fontweight='bold')

plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5: Classification Reports


In [ ]:
for label, preds in [
    ('M1 — Random Forest', pred_m1),
    ('M2 — Logistic Regression', pred_m2),
    (f'M3 — KNN (K={best_k})', pred_m3),
    ('Ensemble', pred_ens),
]:
    print(f'\n{"═"*45}')
    print(f'  {label}')
    print(f'{"═"*45}')
    print(classification_report(Y_test, preds, target_names=['UNDER', 'OVER']))

## Step 6: Feature Importance (M1) & Coefficients (M2)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── M1: Random Forest Feature Importance ──────────────────────────────────
ax = axes[0]
importances = pd.Series(m1.feature_importances_, index=m1_features)
importances = importances.sort_values(ascending=True).tail(15)
bars = ax.barh(importances.index, importances.values, color='steelblue', alpha=0.85, edgecolor='white')
ax.set_xlabel('Feature Importance')
ax.set_title('M1 — Random Forest\nTop 15 Feature Importances', fontweight='bold')
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)

# ── M2: Logistic Regression Coefficients ──────────────────────────────────
ax2 = axes[1]
coefs = pd.Series(m2['model'].coef_[0], index=m2_features).sort_values()
colors_lr = ['tomato' if c < 0 else 'seagreen' for c in coefs.values]
bars2 = ax2.barh(coefs.index, coefs.values, color=colors_lr, alpha=0.85, edgecolor='white')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Coefficient (positive = leans OVER)')
ax2.set_title('M2 — Logistic Regression\nFeature Coefficients', fontweight='bold')
for bar, val in zip(bars2, coefs.values):
    offset = 0.005 if val >= 0 else -0.005
    ax2.text(val + offset, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.show()

## Step 7: Probability Calibration


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, proba, label, color in zip(
    axes,
    [proba_m1, proba_m2, proba_m3],
    ['Random Forest (M1)', 'Logistic Regression (M2)', f'KNN K={best_k} (M3)'],
    ['steelblue', 'tomato', 'seagreen']
):
    try:
        prob_true, prob_pred = calibration_curve(Y_test, proba, n_bins=8, strategy='quantile')
        brier = brier_score_loss(Y_test, proba)
        ax.plot(prob_pred, prob_true, marker='o', color=color, linewidth=2,
                label=f'Calibration (Brier={brier:.3f})')
        ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfect calibration')
        ax.fill_between([0,1],[0,1],[0,1], alpha=0.05, color='gray')
        ax.set_xlabel('Mean Predicted Probability')
        ax.set_ylabel('Fraction of Positives')
        ax.set_title(label, fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_xlim(0,1)
        ax.set_ylim(0,1)
    except Exception as e:
        ax.text(0.5, 0.5, str(e), ha='center', va='center')
        ax.set_title(label)

plt.suptitle('Probability Calibration\n(dots on diagonal = perfectly calibrated)', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('Brier Score interpretation: lower = better calibrated (0.25 = random)')
for label, proba in [('M1', proba_m1), ('M2', proba_m2), ('M3', proba_m3), ('Ensemble', proba_ens)]:
    print(f'  {label}: {brier_score_loss(Y_test, proba):.4f}')

## Step 8: Precision-Recall Curves


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for proba, label, color in [
    (proba_m1,  f'Random Forest  (AP={average_precision_score(Y_test, proba_m1):.3f})',  'steelblue'),
    (proba_m2,  f'Logistic Reg.  (AP={average_precision_score(Y_test, proba_m2):.3f})',  'tomato'),
    (proba_m3,  f'KNN K={best_k}       (AP={average_precision_score(Y_test, proba_m3):.3f})',  'seagreen'),
    (proba_ens, f'Ensemble       (AP={average_precision_score(Y_test, proba_ens):.3f})',  'darkorchid'),
]:
    prec, rec, _ = precision_recall_curve(Y_test, proba)
    ax.plot(rec, prec, label=label, linewidth=2, color=color)

baseline = Y_test.mean()
ax.axhline(baseline, color='gray', linestyle='--', alpha=0.6, label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Step 9: Confidence Distribution


In [ ]:
# How often does each model predict at different confidence levels?
# Higher concentration at extremes (near 0 or 1) = more decisive model

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, proba, label, color in zip(
    axes,
    [proba_m1, proba_m2, proba_m3],
    ['Random Forest (M1)', 'Logistic Regression (M2)', f'KNN K={best_k} (M3)'],
    ['steelblue', 'tomato', 'seagreen']
):
    ax.hist(proba, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(0.5, color='black', linestyle='--', alpha=0.6, label='Decision boundary')
    ax.set_xlabel('Predicted OVER Probability')
    ax.set_ylabel('Count')
    ax.set_title(label, fontweight='bold')
    ax.legend(fontsize=9)
    pct_over = (proba >= 0.5).mean()
    ax.text(0.05, 0.92, f'{pct_over:.1%} predicted OVER',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.suptitle('Predicted Probability Distribution\n(spread = model confidence pattern)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 10: Final Summary


In [ ]:
print('=' * 60)
print('     NBA OVER/UNDER — FULL MODEL EVALUATION SUMMARY')
print('     Season: 2025-26')
print('=' * 60)
print(f'  Test samples: {len(Y_test):,}  |  OVER rate: {Y_test.mean():.2%}')
print()
print(f'  {"Model":<28} {"Acc":>7} {"AUC":>7} {"Brier":>7}')
print('  ' + '-' * 51)

for label, acc, auc, proba in [
    ('Random Forest (M1)',       acc_m1,  auc_m1,  proba_m1),
    ('Logistic Regression (M2)', acc_m2,  auc_m2,  proba_m2),
    (f'KNN K={best_k} (M3)',     acc_m3,  auc_m3,  proba_m3),
    ('Ensemble (M1+M2+M3)',      acc_ens, auc_ens, proba_ens),
]:
    brier = brier_score_loss(Y_test, proba)
    best_acc   = '*' if acc  == max(acc_m1, acc_m2, acc_m3, acc_ens)  else ''
    best_auc   = '*' if auc  == max(auc_m1, auc_m2, auc_m3, auc_ens) else ''
    print(f'  {label:<28} {acc:>7.4f} {auc:>7.4f} {brier:>7.4f}  {best_acc}{best_auc}')

print()
print('  * = best in category')
print()

best_model_auc = ['M1 (RF)', 'M2 (LR)', f'M3 (KNN K={best_k})', 'Ensemble'][np.argmax([auc_m1, auc_m2, auc_m3, auc_ens])]
best_model_acc = ['M1 (RF)', 'M2 (LR)', f'M3 (KNN K={best_k})', 'Ensemble'][np.argmax([acc_m1, acc_m2, acc_m3, acc_ens])]

print(f'  Best AUC:      {best_model_auc}')
print(f'  Best Accuracy: {best_model_acc}')
print()
print('  Recommendation for sports betting:')
print(f'  Use KNN (M3) as primary signal — captures local game similarity')
print(f'  Confirm with prop checker edge before placing any bet')
print('=' * 60)